# Cash User Testing Notebook
Testing the `cash` notebook caching framework in a real JupyterLab environment.

In [ ]:
%load_ext cash
%cash_on
%cash_badge print
%cash_debug on

In [ ]:
import cash
import pandas as pd
import numpy as np
import time
print(f"Cash version: {cash.__version__}")

In [ ]:
t0 = time.time()
np.random.seed(42)
n = 100_000
df = pd.DataFrame({
    'date': pd.date_range('2020-01-01', periods=n, freq='h'),
    'price': np.cumsum(np.random.randn(n)) + 100,
    'volume': np.random.randint(100, 10000, n),
    'category': np.random.choice(['A', 'B', 'C', 'D'], n)
})
elapsed = time.time() - t0
print(f"Created DataFrame: {df.shape} in {elapsed:.2f}s")

In [ ]:
# Heavy computation: rolling statistics
t0 = time.time()
df['rolling_mean'] = df.groupby('category')['price'].transform(
    lambda x: x.rolling(window=100, min_periods=1).mean()
)
df['rolling_std'] = df.groupby('category')['price'].transform(
    lambda x: x.rolling(window=100, min_periods=1).std()
)
df['zscore'] = (df['price'] - df['rolling_mean']) / df['rolling_std'].clip(lower=0.01)
elapsed = time.time() - t0
print(f"Rolling stats computed in {elapsed:.2f}s")
print(f"DataFrame shape: {df.shape}")
print(f"Z-score range: [{df['zscore'].min():.2f}, {df['zscore'].max():.2f}]")

In [ ]:
# Test: Loop with caching
results = {}
for cat in df['category'].unique():
    subset = df[df['category'] == cat]
    results[cat] = {
        'mean_price': subset['price'].mean(),
        'total_volume': subset['volume'].sum(),
        'count': len(subset)
    }
print("Category stats:")
for cat, stats in sorted(results.items()):
    print(f"  {cat}: mean={stats['mean_price']:.2f}, vol={stats['total_volume']:,}, n={stats['count']}")

In [ ]:
# Test: File dependency tracking
import os
print(f"CWD: {os.getcwd()}")
csv_path = os.path.join(os.getcwd(), 'examples', 'sales_data.csv')
if not os.path.exists(csv_path):
    csv_path = 'sales_data.csv'  # try same directory as notebook
print(f"CSV path: {csv_path}, exists: {os.path.exists(csv_path)}")
sales_df = pd.read_csv(csv_path)
print(f"Sales data: {sales_df.shape}")
print(sales_df.head(3).to_string())

In [ ]:
# Test: Conditional logic caching
threshold = 100
if n > 75_000:
    analysis_type = "large"
    sample = df.sample(1000, random_state=42)
else:
    analysis_type = "small"
    sample = df.sample(min(500, len(df)), random_state=42)

print(f"Analysis type: {analysis_type} (n={n}, threshold={threshold})")
print(f"Sample size: {len(sample)}")
print(f"Sample mean price: {sample['price'].mean():.2f}")